# 🧠🤖 第4周·Day6 ⚡ — 本地搭建完整RAG系统

> **学习目标**：动手搭建一个能回答技术文档问题的智能问答系统

### 🛠 技术栈
- Embedding：sentence-transformers
- 向量数据库：ChromaDB
- LLM推理：本地Qwen2-7B（4bit量化）


## 📦 Step 1：加载Embedding模型

我们使用 sentence-transformers 将文本转为向量，这是RAG检索的核心


In [ ]:
# 安装依赖（首次运行）
# !pip install sentence-transformers chromadb transformers torch accelerate

from sentence_transformers import SentenceTransformer

# 加载中文Embedding模型
# 首次运行会自动下载，约400MB
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 测试Embedding
test_texts = ['什么是RAG技术', '检索增强生成是AI领域的重要技术']
embeddings = model.encode(test_texts)

print(f'Embedding维度: {embeddings.shape[1]}')
print(f'相似度: {1 - embeddings[0] @ embeddings[1] / (embeddings[0].__class__(embeddings[0]) * embeddings[1].__class__(embeddings[1])):.4f}')

import numpy as np
from numpy.linalg import norm
sim = np.dot(embeddings[0], embeddings[1]) / (norm(embeddings[0]) * norm(embeddings[1]))
print(f'余弦相似度: {sim:.4f}')


## 📄 Step 2：准备文档数据并分块

将长文档切分成适合检索的小块


In [ ]:
# 准备示例技术文档
documents = {
    "doc1": """
    Transformer架构由Vaswani等人在2017年提出，彻底改变了自然语言处理领域。
    其核心是自注意力机制（Self-Attention），允许模型在处理序列数据时
    同时关注所有位置的信息，解决了传统RNN无法并行计算的问题。
    Transformer由编码器和解码器两部分组成，GPT使用了解码器结构，
    而BERT使用了编码器结构。""",
    
    "doc2": """
    RAG（Retrieval-Augmented Generation）检索增强生成技术，
    由Lewis等人在2020年提出。它将信息检索与文本生成结合，
    先从知识库中检索相关文档片段，再将这些片段作为上下文
    输入给大语言模型，从而生成更准确、更有依据的回答。""",
    
    "doc3": """
    向量数据库是专门为高维向量检索设计的数据库系统。
    常用的向量数据库包括ChromaDB、Milvus、Pinecone、Weaviate等。
    它们支持高效的近似最近邻搜索（ANN），
    能在百万级向量中快速找到最相似的文档。
    常用的距离度量包括余弦相似度、欧氏距离和内积。""",
    
    "doc4": """
    Sentence-Transformers是HuggingFace开源的句子嵌入模型库。
    它可以将任意文本转换为固定维度的向量表示。
    常用的模型包括all-MiniLM-L6-v2和paraphrase模型系列。
    中文场景下推荐使用多语言模型或专门训练的中文模型。
    Embedding的质量直接影响RAG系统的检索效果。""",
    
    "doc5": """
    RLHF（基于人类反馈的强化学习）是让大模型对齐人类价值观的关键技术。
    训练过程分为三步：先进行SFT（监督微调），然后训练奖励模型，
    最后用PPO算法进行策略优化。DPO是RLHF的简化版本，
    直接用偏好数据优化模型，无需训练奖励模型。"""
}

print(f'共加载 {len(documents)} 篇文档')


In [ ]:
# 文档分块处理
def chunk_text(text, chunk_size=200, overlap=50):
    """将长文本分块，支持重叠"""
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # 重叠部分
    return chunks

# 对所有文档进行分块
all_chunks = []
chunk_metadata = []

for doc_id, text in documents.items():
    chunks = chunk_text(text, chunk_size=150, overlap=30)
    all_chunks.extend(chunks)
    chunk_metadata.extend([{'doc_id': doc_id, 'chunk_idx': i} for i in range(len(chunks))])

print(f'总共生成 {len(all_chunks)} 个文档块')
for i, (chunk, meta) in enumerate(zip(all_chunks[:3], chunk_metadata[:3])):
    print(f'\nChunk {i+1} ({meta["doc_id"]}): {chunk[:60]}...')


## 🔢 Step 3：生成向量嵌入并存入ChromaDB


In [ ]:
import chromadb

# 初始化ChromaDB客户端（内存模式）
chroma_client = chromadb.Client()

# 创建或获取集合
collection_name = 'tech_docs_rag'
collection = chroma_client.get_or_create_collection(name=collection_name)

# 生成所有文档块的向量嵌入
embeddings = model.encode(all_chunks)
print(f'嵌入向量形状: {embeddings.shape}')

# 准备ChromaDB格式的ID
chunk_ids = [f'{meta["doc_id"]}_{meta["chunk_idx"]}' for meta in chunk_metadata]

# 存入ChromaDB
collection.add(
    ids=chunk_ids,
    documents=all_chunks,
    embeddings=embeddings.tolist()
)

print(f'\n成功存入 {len(all_chunks)} 个文档块到 ChromaDB')
print(f'集合名称: {collection_name}')


## 🔍 Step 4：实现RAG检索生成函数


In [ ]:
# RAG检索函数
def rag_retrieve(query, top_k=3):
    """检索最相关的文档块"""
    # 生成查询向量
    query_embedding = model.encode([query])
    
    # 在ChromaDB中检索
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    
    return results

# 测试检索
query = '什么是Transformer架构？'
results = rag_retrieve(query, top_k=3)

print(f'🔍 查询: {query}')
print(f'\n检索结果（Top-{3}）：')
for i in range(len(results['ids'][0])):
    doc_id = results['ids'][0][i]
    distance = results['distances'][0][i]
    doc = results['documents'][0][i]
    print(f'\n[{i+1}] {doc_id} (距离: {distance:.4f})')
    print(f'    {doc[:80]}...')


In [ ]:
# 构建完整RAG问答函数
def rag_qa(question, top_k=3):
    """完整的RAG问答流程"""
    # Step 1: 检索相关文档
    results = rag_retrieve(question, top_k=top_k)
    
    # Step 2: 组装上下文
    context_parts = []
    for i, doc in enumerate(results['documents'][0]):
        context_parts.append(f'[文档{i+1}] {doc}')
    context = '\n\n'.join(context_parts)
    
    # Step 3: 构建Prompt
    prompt = f"""请基于以下参考资料回答问题。如果资料中没有相关信息，请说明。

参考资料：
{context}

问题：{question}

回答："""
    
    return prompt, results

# 测试完整RAG流程
questions = [
    '什么是RAG技术？',
    'Transformer解决了什么问题？',
    '向量数据库有哪些？'
]

for q in questions:
    print(f'\n{'='*50}')
    prompt, results = rag_qa(q)
    print(f'❓ 问题: {q}')
    print(f'📎 检索到 {len(results["ids"][0])} 个相关文档块')
    for i, (doc_id, dist) in enumerate(zip(results['ids'][0], results['distances'][0])):
        print(f'   {i+1}. {doc_id} (距离: {dist:.4f})')


## 📊 Step 5：测试系统性能


In [ ]:
# 测试多个问题并分析检索质量
test_questions = [
    {'q': '什么是自注意力机制？', 'expected': ['doc1']},
    {'q': 'RLHF的训练步骤是什么？', 'expected': ['doc5']},
    {'q': '常用的向量数据库有哪些？', 'expected': ['doc3']},
    {'q': '如何提高Embedding质量？', 'expected': ['doc4']},
    {'q': 'RAG技术由谁提出？', 'expected': ['doc2']},
]

hits = 0
total = len(test_questions)

print('📊 检索质量评估：')
print('=' * 60)

for test in test_questions:
    results = rag_retrieve(test['q'], top_k=2)
    retrieved_ids = [rid.split('_')[0] for rid in results['ids'][0]]
    hit = any(exp in retrieved_ids for exp in test['expected'])
    hits += int(hit)
    status = '✅' if hit else '❌'
    print(f'{status} Q: {test["q"]}')
    print(f'   Expected: {test["expected"]}, Retrieved: {retrieved_ids}')

print(f'\n🎯 召回率: {hits}/{total} = {hits/total*100:.1f}%')


In [ ]:
# 可视化检索结果分布
fig, ax = plt.subplots(figsize=(10, 5))

doc_counts = {}
for test in test_questions:
    results = rag_retrieve(test['q'], top_k=3)
    for rid in results['ids'][0]:
        doc = rid.split('_')[0]
        doc_counts[doc] = doc_counts.get(doc, 0) + 1

docs = list(doc_counts.keys())
counts = list(doc_counts.values())
colors = ['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4', '#FFEAA7']

bars = ax.bar(docs, counts, color=colors[:len(docs)])
ax.set_xlabel('Document ID')
ax.set_ylabel('Retrieval Count')
ax.set_title('Document Retrieval Distribution')

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第4周/day6_retrieval_dist.png', dpi=150)
print('Chart saved to day6_retrieval_dist.png')


## 💡 进阶优化方向

1. **混合检索**：向量检索 + BM25关键词检索，取交集
2. **重排序**：用Cross-Encoder对检索结果重新打分
3. **查询改写**：用HyDE生成假设答案，提升检索相关性
4. **动态Top-K**：根据问题复杂度自动调整检索数量


## ✅ 验证清单

- [ ] ChromaDB能成功存入和检索文档块
- [ ] 检索结果与问题相关（召回率 > 80%）
- [ ] 不同问题能命中不同文档
- [ ] 相似度分数合理分布

## 💡 业务关联思考

如果用这套RAG系统来管理糖水店的经营知识库：
- 存入所有配方文档，顾客问"有什么甜品"就能精准推荐
- 存入进货记录，问"上周进了什么材料"直接检索回答
- 存入价格信息，快速查询某款产品的定价策略


## 🔑 今日英文术语

- **Embedding Model** [ˈɛmbɛdɪŋ ˈmɒdəl] 嵌入模型
- **Vector Store** [ˈvɛktər stɔːr] 向量存储
- **Nearest Neighbor** [ˈnɪərɪst ˈneɪbər] 最近邻搜索
- **Semantic Search** [sɪˈmæntɪk sɜːrtʃ] 语义搜索
- **Document Chunking** [ˈdɒkjʊmənt ˈtʃʌŋkɪŋ] 文档分块


## 🎬 推荐视频

【从零搭建RAG系统】（30分钟）
https://www.bilibili.com/video/BV1Px4y1b7Jx

## 📖 延伸阅读

HuggingFace文档: https://huggingface.co/docs/transformers
ChromaDB文档: https://www.trychroma.com/
